# Encrypted Machine Learning: Core (`ml/core.py`)

This tutorial covers `src/concrete_fhe_toolkit/ml/core.py`. This module provides low-level mathematical operations essential for machine learning, such as distance calculations (Manhattan, Hamming, Euclidean) and base loss/metric calculations.

## 1. Distance Metrics

Let's test `manhattan_distance`, `hamming_distance`, and `euclidean_distance_squared` between two encrypted vectors.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.core import (
    manhattan_distance, hamming_distance, euclidean_distance_squared
)

def test_distances(x1: int, y1: int, x2: int, y2: int):
    v1 = [x1, y1]
    v2 = [x2, y2]
    return (
        manhattan_distance(v1, v2),
        hamming_distance(v1, v2),
        euclidean_distance_squared(v1, v2)
    )

compiler = fhe.Compiler(test_distances, {
    "x1": "encrypted", "y1": "encrypted", 
    "x2": "encrypted", "y2": "encrypted"
})

inputset = [(0, 0, 3, 4), (1, 1, 1, 1)]
circuit = compiler.compile(inputset)

# Vector 1: [0, 0], Vector 2: [3, 4]
# Manhattan: |0-3| + |0-4| = 7
# Hamming: (0!=3) + (0!=4) = 2 mismatches
# Euclidean Sq: (0-3)^2 + (0-4)^2 = 9 + 16 = 25
man, ham, euc_sq = circuit.encrypt_run_decrypt(0, 0, 3, 4)

assert man == 7
assert ham == 2
assert euc_sq == 25

print("✅ Encrypted Distance metrics passed!")

## 2. Confusion Matrix Components

The core module also exports secure primitives for confusion matrix calculations (`true_positives`, `false_negatives`, etc.).

In [ ]:
from concrete_fhe_toolkit.ml.core import confusion_matrix

def test_conf_matrix(p1: int, p2: int, p3: int, t1: int, t2: int, t3: int):
    return confusion_matrix([p1, p2, p3], [t1, t2, t3])

compiler = fhe.Compiler(test_conf_matrix, {
    "p1": "encrypted", "p2": "encrypted", "p3": "encrypted",
    "t1": "encrypted", "t2": "encrypted", "t3": "encrypted"
})
circuit = compiler.compile([(1, 0, 1, 1, 1, 0)])

# Preds: [1, 0, 1]
# Trues: [1, 1, 0]
# TP (1&1): 1
# TN (0&0): 0
# FP (1&0): 1
# FN (0&1): 1
matrix = circuit.encrypt_run_decrypt(1, 0, 1, 1, 1, 0)
tn, fp = matrix[0]
fn, tp = matrix[1]

assert tn == 0
assert fp == 1
assert fn == 1
assert tp == 1

print("✅ Encrypted Confusion Matrix passed!")